<a href="https://colab.research.google.com/github/joseomarleal-maker/bot-trading/blob/main/Copia_de_Copia_de_Untitled1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install yfinance pandas pandas-ta

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.3/240.3 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 16.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: llvmlite
    Found existing installation: llvmlite 0.43.0
    Uninstalling llvmlite-0.43.0:
      Successfully uninstalled llvmlite-0.43.0
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninsta

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import yfinance as yf
import pandas_ta as ta
import pandas as pd
import time
import requests
from datetime import datetime

# 🔴 CONFIGURACIÓN DE TU TELEGRAM
TOKEN_TELEGRAM = "8451771652:AAGaRcMMipizwj1zj95oZCzu51-Zn1J1To4"
ID_CHAT = "6600690280"

# 📊 TUS DIVISAS DE INVESTING
MERCADOS = ["EURUSD=X", "EURGBP=X", "EURJPY=X", "EURCHF=X", "EURCAD=X", "EURAUD=X", "EURNZD=X"]

def enviar_alerta_telegram(mensaje):
    url = f"https://api.telegram.org/bot{TOKEN_TELEGRAM}/sendMessage"
    payload = {"chat_id": ID_CHAT, "text": mensaje}
    try: requests.post(url, json=payload)
    except Exception as e: print(f"⚠️ Error Telegram: {e}")

print("🛡️ BOT DE ALTA PRECISIÓN ACTIVADO (FILTROS ESTRICTOS)...")
print("="*60)

while True:
    hora_actual = datetime.now().strftime('%H:%M:%S')

    for activo in MERCADOS:
        try:
            nombre_bonito = activo.replace("=X", "")
            nombre_bonito = f"{nombre_bonito[:3]}/{nombre_bonito[3:]}"

            # Descarga de datos
            datos = yf.download(activo, period="2d", interval="1m", progress=False)
            if datos.empty or len(datos) < 50: continue

            if isinstance(datos.columns, pd.MultiIndex):
                datos.columns = datos.columns.droplevel(1)
            datos.columns = datos.columns.str.lower()

            # 🔥 INDICADORES DE ALTA PRECISIÓN
            datos['ema_50'] = ta.ema(datos['close'], length=50) # Tendencia estructural
            datos['rsi'] = ta.rsi(datos['close'], length=14)    # Fuerza del precio

            # Bandas de Bollinger (Desviación 2 para capturar extremos reales)
            bbands = ta.bbands(datos['close'], length=20, std=2)
            datos['bb_lower'] = bbands['BBL_20_2.0']
            datos['bb_upper'] = bbands['BBU_20_2.0']

            ultima_vela = datos.iloc[-1]
            precio = ultima_vela['close']
            rsi = ultima_vela['rsi']
            ema = ultima_vela['ema_50']
            bb_inf = ultima_vela['bb_lower']
            bb_sup = ultima_vela['bb_upper']

            # 🟢 FILTRO DE COMPRA ULTRA ESTRICTO:
            # Tendencia alcista + RSI en zona de colapso (< 30) + Precio rompiendo banda inferior
            if precio > ema and rsi <= 30 and precio <= bb_inf:
                msg = f"🚨 ALERTA PREMIUM: COMPRA (SUBE) 🟢\n\n🎯 Divisa: {nombre_bonito}\nPrecio: {precio:.5f}\n⚡ RSI Extremo: {rsi:.2f}\n📊 Confirmación: Rebote en Banda Inferior BB\n\n🎯 Operación recomendada a 1-2 minutos en IQ Option."
                print(f"[{hora_actual}] ✔️ {nombre_bonito}: SEÑAL ENVIADA")
                enviar_alerta_telegram(msg)
                time.sleep(10)

            # 🔴 FILTRO DE VENTA ULTRA ESTRICTO:
            # Tendencia bajista + RSI sobrecomprado (> 70) + Precio rompiendo banda superior
            elif precio < ema and rsi >= 70 and precio >= bb_sup:
                msg = f"🚨 ALERTA PREMIUM: VENTA (BAJA) 🔴\n\n🎯 Divisa: {nombre_bonito}\nPrecio: {precio:.5f}\n⚡ RSI Extremo: {rsi:.2f}\n📊 Confirmación: Rechazo en Banda Superior BB\n\n🎯 Operación recomendada a 1-2 minutos en IQ Option."
                print(f"[{hora_actual}] ✔️ {nombre_bonito}: SEÑAL ENVIADA")
                enviar_alerta_telegram(msg)
                time.sleep(10)

        except Exception as e:
            pass

    time.sleep(15)

🛡️ BOT DE ALTA PRECISIÓN ACTIVADO (FILTROS ESTRICTOS)...


In [ ]:
import requests

# ¡TOKEN REAL CORREGIDO! 🎯
TOKEN_TELEGRAM = "8451771652:AAGaRcMMipizwj12j95oZCzu51-Zn1J1To4"
ID_CHAT = "6600690280"

url = f"https://api.telegram.org/bot{TOKEN_TELEGRAM}/sendMessage"
payload = {
    "chat_id": ID_CHAT,
    "text": "¡💥 ¡SÍ SE PUDO, JOSÉ! Sincronización completada al 100%. Tu bot de Python ya está conectado a tu celular con éxito. 🚀🤖"
}

print("📡 Intentando enlace definitivo con el token real...")
respuesta = requests.post(url, json=payload)
print("Respuesta del servidor:", respuesta.json())

📡 Intentando enlace definitivo con el token real...
Respuesta del servidor: {'ok': True, 'result': {'message_id': 4, 'from': {'id': 8451771652, 'is_bot': True, 'first_name': 'Mi Alerta Trading', 'username': 'robot_trading2026_bot'}, 'chat': {'id': 6600690280, 'first_name': 'Jose', 'type': 'private'}, 'date': 1783082551, 'text': '¡💥 ¡SÍ SE PUDO, JOSÉ! Sincronización completada al 100%. Tu bot de Python ya está conectado a tu celular con éxito. 🚀🤖'}}
